#### 1) First page 

In [1]:
from bs4 import BeautifulSoup
import pandas as pd
import requests

# Base URL for Quotes to Scrape
url = "http://quotes.toscrape.com/"
response = requests.get(url)

# Parse the HTML content
soup = BeautifulSoup(response.text, "html.parser")

# Find all quote containers on the page
quote_boxes = soup.find_all("div", class_="quote")

quotes_data = []

for box in quote_boxes:
  text = box.find("span", class_="text").get_text()
  author = box.find("small", class_="author").get_text()
  tags = [tag.get_text() for tag in box.find_all("a", class_="tag")]

  quotes_data.append({"Quote": text, "Author": author, "Tags": ", ".join(tags)})

# Convert into a Pandas DataFrame
df = pd.DataFrame(quotes_data)
df.head()

,Quote,Author,Tags
0,“The world as we have created it is a process ...,Albert Einstein,"change, deep-thoughts, thinking, world"
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling,"abilities, choices"
2,“There are only two ways to live your life. On...,Albert Einstein,"inspirational, life, live, miracle, miracles"
3,"“The person, be it gentleman or lady, who has ...",Jane Austen,"aliteracy, books, classic, humor"
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe,"be-yourself, inspirational"


#### 2Scraping Multiple Pages (Pagination)
Because quotes span across multiple pages, you can write a loop that automatically navigates through them until no more pages remain:

In [2]:
all_quotes = []
page = 1

while True:
  url = f"http://quotes.toscrape.com/page/{page}/"
  response = requests.get(url)

  # If the page doesn't exist (e.g., 404 error), break the loop
  if response.status_code != 200:
    break

  soup = BeautifulSoup(response.text, "html.parser")
  quote_boxes = soup.find_all("div", class_="quote")

  for box in quote_boxes:
    text = box.find("span", class_="text").get_text()
    author = box.find("small", class_="author").get_text()
    tags = [tag.get_text() for tag in box.find_all("a", class_="tag")]
    all_quotes.append({"Quote": text, "Author": author, "Tags": ", ".join(tags)})

  # Check if a "Next" button exists; if not, you've reached the last page
  next_btn = soup.find("li", class_="next")
  if not next_btn:
    break

  page += 1

# Convert full multi-page dataset to DataFrame and save
df_all = pd.DataFrame(all_quotes)
df_all.to_csv("quotes_scraped.csv", index=False, encoding="utf-8")
print(f"Successfully scraped {len(df_all)} quotes across {page} pages!")

Successfully scraped 100 quotes across 10 pages!
